# 19.16 Uncertainty quantification & calibration

Calibration asks whether predictions with probability 0.8 are correct about 80% of the time.

Accuracy says whether labels are right; calibration says whether the probabilities have frequency meaning. We compute reliability curves and ECE across the ladder. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_wine, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_scaled_logistic(X, y, test_size=0.4, random_state=0):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_te, y_tr, y_te


def fit_scaled_logistic_three_way(X, y, random_state=0):
    if len(y) < 12:
        repeats = int(np.ceil(12 / len(y)))
        X = np.tile(X, (repeats, 1))
        y = np.tile(y, repeats)
    x_tmp, x_te, y_tmp, y_te = train_test_split(X, y, test_size=0.3, random_state=random_state, stratify=y)
    x_tr, x_cal, y_tr, y_cal = train_test_split(x_tmp, y_tmp, test_size=0.35, random_state=random_state, stratify=y_tmp)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_cal = scaler.transform(x_cal)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_cal, x_te, y_tr, y_cal, y_te


def top_class_binary_view(y, proba):
    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y).astype(int)
    return correct, conf, pred


def degrade_d5_features(name, X):
    rng = np.random.default_rng(1918)
    if name.startswith("D5"):
        return X + rng.normal(0.0, X.std(axis=0) * 0.35, size=X.shape)
    return X


## The concept, built once

Expected calibration error bins confidence and compares empirical accuracy to confidence:

$$ECE=\sum_b \frac{n_b}{n}|acc(b)-conf(b)|.$$

The lesson ECE components are $0.100,0.050,0.200$, so total must be $0.350$ and guard must be $0.420$.

In [ ]:

def calibration_report(y, proba, bins=10):
    correct, conf, pred = top_class_binary_view(y, proba)
    edges = np.linspace(0.0, 1.0, bins + 1)
    rows = []
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi)
        if hi == 1.0:
            mask = (conf > lo) & (conf <= hi)
        if not np.any(mask):
            rows.append((lo, hi, 0, np.nan, np.nan, 0.0))
            continue
        acc = float(correct[mask].mean())
        avg_conf = float(conf[mask].mean())
        weight = float(mask.mean())
        gap = abs(acc - avg_conf)
        ece += weight * gap
        rows.append((lo, hi, int(mask.sum()), acc, avg_conf, weight * gap))
    return float(ece), rows

components = np.array([0.100, 0.050, 0.200])
total = float(components.sum())
abs_mass = float(np.abs(components).sum())
share = float(abs(components[0]) / abs_mass)
guarded = total + 0.2 * abs_mass
contrast = total - components[2]
change = contrast - total

assert np.isclose(total, 0.350)
assert np.isclose(share, 0.286, atol=0.001)
assert np.isclose(guarded, 0.420)
assert np.isclose(change, -0.200)
print("total", total, "guard", guarded, "share", share)


The reusable rung function trains one classifier, takes the top-class confidence, and reports ECE plus reliability points. Temperature scaling is included as the repair knob.

In [ ]:

def temperature_scale(proba, temperature):
    clipped = np.clip(proba, 1e-9, 1.0)
    logits = np.log(clipped)
    scaled = logits / temperature
    shifted = scaled - scaled.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def rung_calibration(X, y, bins=10, temperature=1.0):
    clf, x_tr, x_te, y_tr, y_te = fit_scaled_logistic(X, y)
    proba = clf.predict_proba(x_te)
    scaled = temperature_scale(proba, temperature)
    ece, rows = calibration_report(y_te, scaled, bins=bins)
    acc = accuracy_score(y_te, scaled.argmax(axis=1))
    return ece, acc, rows, y_te, scaled


## The dataset ladder

All six notebooks in this batch use the same F15 classification ladder: a hand-checkable D1 toy, synthetic rungs, two real tabular rungs, and the real D5 Breast Cancer stress rung. The method changes, not the data-loading story.

In [ ]:

rungs = clf_ladder()

for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    preview = np.round(X[:3, : min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  classes:", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample columns:")
    print(preview)


In [ ]:

rows = []
for rung, (name, X, y) in enumerate(rungs, start=1):
    ece, acc, bins, y_te, proba = rung_calibration(X, y, bins=10)
    rows.append((rung, name, ece, acc))

print("rung | ECE | accuracy")
for rung, name, ece, acc in rows:
    print(f"D{rung} | {ece:.3f} | {acc:.3f} | {name}")


In [ ]:

summary = []
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for col, (name, X, y) in enumerate(rungs):
    ece, acc, bin_rows, y_te, proba = rung_calibration(X, y, bins=8)
    summary.append(ece)
    xs = [row[4] for row in bin_rows if row[2] > 0]
    ys = [row[3] for row in bin_rows if row[2] > 0]
    axes[col].plot([0, 1], [0, 1], color="gray", linestyle="--")
    axes[col].scatter(xs, ys)
    axes[col].set_title(name.split("(")[0].strip(), fontsize=9)
    axes[col].set_xlabel("confidence")
    axes[col].set_ylabel("accuracy")

plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), summary, marker="o")
plt.xlabel("ladder rung")
plt.ylabel("ECE")
plt.title("ECE vs. D1→D5")
plt.xticks(range(1, 6))
plt.show()


## Pitfall on D5: looking only at accuracy

D5 can have a strong accuracy while being overconfident. The fix reports reliability diagrams, ECE, and a bin-count sensitivity check instead of stopping at accuracy.

In [ ]:

name, X, y = rungs[-1]
ece_5, acc_5, bins_5, y_te, proba = rung_calibration(X, y, bins=5)
ece_20, acc_20, bins_20, y_te, proba = rung_calibration(X, y, bins=20)
cooled_ece, cooled_acc, cooled_bins, y_te, cooled = rung_calibration(X, y, bins=10, temperature=1.5)

print("accuracy only", acc_5)
print("ECE with 5 bins", ece_5)
print("ECE with 20 bins", ece_20)
print("temperature-scaled ECE", cooled_ece)


## Evaluate it + Practice

- Metric: ECE across all rungs.
- No-skill baseline: report only held-out accuracy.
- Cheap sanity check: a perfectly calibrated synthetic confidence vector has ECE near zero.
- Ablation: use temperature 0.5 to induce overconfidence.
- Failure signals: accuracy high but ECE high, noisy bins, or discarded probabilities.

Practice 1: Change the random seed or stress strength and explain which rung moves most.

Practice 2: Try 5, 10, and 20 bins on D3 and compare stability.

Practice 3: Apply temperature scaling to D5 and choose the best temperature on a calibration split.